# LoRA Fine-Tuning Small LMs for Mathematical Reasoning

This notebook implements Parameter-Efficient Fine-Tuning (LoRA) on three small language models:
- **TinyLlama-1.1B-Chat**
- **Qwen2.5-1.5B-Instruct**
- **OpenLlama-3B-v2**

Evaluated on: GSM8K, SVAMP, AQuA-RAT (300 test samples each)

## 1. Install Dependencies

In [ ]:
!pip install -q transformers>=4.40.0 peft>=0.10.0 datasets accelerate bitsandbytes trl scipy sentencepiece protobuf

import torch
print(f"PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

PyTorch: 2.10.0+cu128, CUDA available: True


## 2. Upload & Load Data

Upload your `processed.zip` to Colab, then run the cell below.

In [ ]:
import os, json, zipfile
from google.colab import files

from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/processed.zip /content/

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile("/content/processed.zip", 'r') as z:
    z.extractall(DATA_DIR)

print("Extracted files:")
for f in sorted(os.listdir(DATA_DIR)):
    print(f"  {f}")

Mounted at /content/drive
Extracted files:
  aqua_rat_test.jsonl
  aqua_rat_train.jsonl
  aqua_rat_validation.jsonl
  gsm8k_test.jsonl
  gsm8k_train.jsonl
  gsm8k_validation.jsonl
  merged_test.jsonl
  merged_train.jsonl
  merged_validation.jsonl
  svamp_test.jsonl
  svamp_train.jsonl
  svamp_validation.jsonl


In [ ]:
def load_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

# Load all splits
train_data = load_jsonl(os.path.join(DATA_DIR, "merged_train.jsonl"))
val_data = load_jsonl(os.path.join(DATA_DIR, "merged_validation.jsonl"))

# Load individual test sets for per-dataset evaluation
test_gsm8k = load_jsonl(os.path.join(DATA_DIR, "gsm8k_test.jsonl"))
test_svamp = load_jsonl(os.path.join(DATA_DIR, "svamp_test.jsonl"))
test_aqua = load_jsonl(os.path.join(DATA_DIR, "aqua_rat_test.jsonl"))

print(f"Train: {len(train_data)} | Val: {len(val_data)}")
print(f"Test - GSM8K: {len(test_gsm8k)} | SVAMP: {len(test_svamp)} | AQuA-RAT: {len(test_aqua)}")

Train: 79183 | Val: 8797
Test - GSM8K: 300 | SVAMP: 300 | AQuA-RAT: 300


## 3. Format Data for Instruction Tuning

Convert each sample into a prompt-completion pair with Chain-of-Thought style.

In [ ]:
def format_prompt(sample):
    """Build the instruction prompt (input to model)."""
    question = sample["question"].strip()
    prompt = (
        "Solve the following math problem step by step.\n"
        "Show your reasoning, then give the final answer after 'The answer is: '.\n\n"
        f"Question: {question}\n\n"
        "Solution:"
    )
    return prompt


def format_completion(sample):
    """Build the expected completion (target output)."""
    rationale = sample.get("rationale", "") or ""
    answer = sample["answer"].strip()
    # Clean up rationale
    rationale = rationale.strip().rstrip('"').strip()
    if rationale:
        return f" {rationale}\nThe answer is: {answer}"
    else:
        return f" The answer is: {answer}"


# Quick sanity check
sample = train_data[0]
print("=== PROMPT ===")
print(format_prompt(sample))
print("\n=== COMPLETION ===")
print(format_completion(sample))

=== PROMPT ===
Solve the following math problem step by step.
Show your reasoning, then give the final answer after 'The answer is: '.

Question: In a friendship gang Aravind has 2 gang , in how many ways can he invite one or more of the gang to his house ?

Solution:

=== COMPLETION ===
 Aravind can select one or more than one of his 8 gang . = > Required number of ways = 2 ^ 2 – 1 = 3 . C Therefore, the correct answer is: 3
The answer is: 3


## 4. Subsample Training Data (Important for Colab!)

The merged training set has ~79K samples. On Colab T4, training on all of them would take too long.  
We subsample strategically: keep a balanced mix from all 3 datasets.

In [ ]:
import random
random.seed(42)


TRAIN_BUDGET = 5000

# Split by dataset
train_by_ds = {}
for s in train_data:
    ds = s["dataset"]
    train_by_ds.setdefault(ds, []).append(s)

print("Full training set distribution:")
for ds, items in train_by_ds.items():
    print(f"  {ds}: {len(items)}")

# Balanced sampling: equal from each dataset (or all if smaller than budget/3)
per_ds = TRAIN_BUDGET // len(train_by_ds)
train_subset = []
for ds, items in train_by_ds.items():
    n = min(per_ds, len(items))
    train_subset.extend(random.sample(items, n))

random.shuffle(train_subset)

# Subsample validation too
val_subset = random.sample(val_data, min(500, len(val_data)))

print(f"\nSubsampled: {len(train_subset)} train, {len(val_subset)} val")

Full training set distribution:
  aqua_rat: 71827
  svamp: 630
  gsm8k: 6726

Subsampled: 3962 train, 500 val


## 5. Model & LoRA Configuration



In [30]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ============================================================
#  MODEL CONFIGS - switch by changing ACTIVE_MODEL
# ============================================================
MODEL_CONFIGS = {
    "tinyllama": {
        "model_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "target_modules": ["q_proj", "v_proj"],
        "load_in_4bit": False,   # 1.1B fits in fp16 on T4
    },
    "qwen2.5": {
        "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
        "target_modules": ["q_proj", "v_proj"],
        "load_in_4bit": False,   # 1.5B fits in fp16 on T4
    },
    "openllama3b": {
        "model_name": "openlm-research/open_llama_3b_v2",
        "target_modules": ["q_proj", "v_proj"],
        "load_in_4bit": True,    # 3B needs 4-bit quantization on T4
    },
}

ACTIVE_MODEL = "tinyllama"
# >>>>> Options: "tinyllama", "qwen2.5", "openllama3b" <<<<<

cfg = MODEL_CONFIGS[ACTIVE_MODEL]
print(f"Active model: {cfg['model_name']}")
print(f"4-bit quantization: {cfg['load_in_4bit']}")

Active model: Qwen/Qwen2.5-1.5B-Instruct
4-bit quantization: False


In [ ]:
# ============================================================
#  Load Tokenizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    cfg["model_name"],
    trust_remote_code=True,
)

# Ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size: 151643
Pad token: <|endoftext|> (id=151643)


In [ ]:
# ============================================================
#  Load Base Model
# ============================================================
load_kwargs = {
    "trust_remote_code": True,
    "device_map": "auto",
}

load_kwargs["torch_dtype"] = torch.float16

base_model = AutoModelForCausalLM.from_pretrained(cfg["model_name"], **load_kwargs)

# Print model size info
total_params = sum(p.numel() for p in base_model.parameters())

print(f"Base model loaded: {total_params / 1e6:.1f}M parameters")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded: 1543.7M parameters
GPU memory: 5.02 GB


In [ ]:
# ============================================================
#  Apply LoRA
# ============================================================
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=cfg["target_modules"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5-1% of total parameters are trainable

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


## 6. Build HuggingFace Dataset for Trainer

In [ ]:
from datasets import Dataset

MAX_LENGTH = 512  # adjust if your samples are longer

def tokenize_sample(sample):
    """Tokenize a single sample into input_ids + labels for causal LM training.

    The prompt part has labels = -100 (ignored in loss).
    Only the completion part contributes to the loss.
    """
    prompt = format_prompt(sample)
    completion = format_completion(sample)
    full_text = prompt + completion

    # Tokenize full text
    full_enc = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    # Tokenize prompt only to find where completion starts
    prompt_enc = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    prompt_len = len(prompt_enc["input_ids"])

    # Build labels: -100 for prompt tokens, actual ids for completion tokens
    labels = [-100] * prompt_len + full_enc["input_ids"][prompt_len:]

    return {
        "input_ids": full_enc["input_ids"],
        "attention_mask": full_enc["attention_mask"],
        "labels": labels,
    }


# Process datasets
print("Tokenizing training data...")
train_tokenized = [tokenize_sample(s) for s in train_subset]
print("Tokenizing validation data...")
val_tokenized = [tokenize_sample(s) for s in val_subset]

train_dataset = Dataset.from_list(train_tokenized)
val_dataset = Dataset.from_list(val_tokenized)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

# Check token length distribution
lengths = [len(x["input_ids"]) for x in train_tokenized]
print(f"Token lengths - min: {min(lengths)}, max: {max(lengths)}, mean: {sum(lengths)/len(lengths):.0f}")

Tokenizing training data...
Tokenizing validation data...
Train dataset: 3962 samples
Val dataset: 500 samples
Token lengths - min: 68, max: 512, mean: 186


## 7. Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import time

OUTPUT_DIR = f"/content/lora_{ACTIVE_MODEL}"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch size = 4 * 4 = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",                   # set to "wandb" if you want logging
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print(f"Starting training for {ACTIVE_MODEL}...")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training for tinyllama...
  Epochs: 3
  Effective batch size: 16
  Learning rate: 0.0002
  LoRA rank: 16, alpha: 32


In [31]:
# ============================================================
#  Train & Record Metrics
# ============================================================
torch.cuda.reset_peak_memory_stats()
start_time = time.time()

train_result = trainer.train()

train_time = time.time() - start_time
peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3

print(f"\n{'='*50}")
print(f"Training completed for: {ACTIVE_MODEL}")
print(f"  Total time: {train_time/60:.1f} minutes")
print(f"  Peak GPU memory: {peak_memory_gb:.2f} GB")
print(f"  Final train loss: {train_result.training_loss:.4f}")
print(f"{'='*50}")

Step,Training Loss,Validation Loss
200,0.862874,1.107302
400,0.775210,1.098067
600,0.730860,1.096673



Training completed for: qwen2.5
  Total time: 3.1 minutes
  Peak GPU memory: 9.14 GB
  Final train loss: 0.8019


In [32]:
# Save the LoRA adapter
SAVE_PATH = f"/content/lora_adapter_{ACTIVE_MODEL}"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"LoRA adapter saved to: {SAVE_PATH}")

# Also save training metrics
metrics = {
    "model": cfg["model_name"],
    "active_model": ACTIVE_MODEL,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "train_samples": len(train_subset),
    "epochs": int(training_args.num_train_epochs),
    "effective_batch_size": training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
    "learning_rate": training_args.learning_rate,
    "train_loss": round(train_result.training_loss, 4),
    "train_time_minutes": round(train_time / 60, 1),
    "peak_gpu_memory_gb": round(peak_memory_gb, 2),
}

with open(f"/content/train_metrics_{ACTIVE_MODEL}.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

LoRA adapter saved to: /content/lora_adapter_qwen2.5
{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "active_model": "qwen2.5",
  "lora_r": 16,
  "lora_alpha": 32,
  "train_samples": 3962,
  "epochs": 3,
  "effective_batch_size": 16,
  "learning_rate": 0.0002,
  "train_loss": 0.8019,
  "train_time_minutes": 3.1,
  "peak_gpu_memory_gb": 9.14
}


## 8. Evaluation on Test Sets

Generate answers and compute accuracy for each dataset.

In [33]:
import re

def extract_answer(text):
    """Extract the final answer from model output.
    Looks for 'The answer is: X' pattern first, then falls back to last number.
    """
    text = text.strip()

    # Pattern 1: "The answer is: X"
    match = re.search(r'[Tt]he answer is[:\s]+([^\n]+)', text)
    if match:
        return match.group(1).strip().rstrip('.')

    # Pattern 2: "#### X" (GSM8K style)
    match = re.search(r'####\s*(.+)', text)
    if match:
        return match.group(1).strip()

    # Fallback: last number in text
    numbers = re.findall(r'-?[\d,]+\.?\d*', text)
    if numbers:
        return numbers[-1].replace(',', '')

    return text.strip()


def normalize_answer(ans):
    """Normalize answer string for comparison."""
    ans = str(ans).strip().lower()
    ans = ans.replace(',', '').replace('$', '').replace('%', '').rstrip('.')
    # Try to convert to float for numeric comparison
    try:
        return str(float(ans))
    except ValueError:
        return ans


def evaluate_model(model, tokenizer, test_data, dataset_name, max_new_tokens=256, batch_size=1):
    """Evaluate model on a test set and return accuracy."""
    model.eval()
    correct = 0
    total = len(test_data)
    results = []

    for i, sample in enumerate(test_data):
        prompt = format_prompt(sample)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,          # greedy decoding for reproducibility
                temperature=1.0,
                pad_token_id=tokenizer.pad_token_id,
            )

        # Decode only the generated part
        generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        pred = extract_answer(generated)
        gold = sample["answer"]

        is_correct = normalize_answer(pred) == normalize_answer(gold)
        if is_correct:
            correct += 1

        results.append({
            "id": sample["id"],
            "question": sample["question"],
            "gold": gold,
            "pred": pred,
            "generated_text": generated[:300],
            "correct": is_correct,
        })

        if (i + 1) % 50 == 0:
            print(f"  [{dataset_name}] {i+1}/{total} done, running acc: {correct/(i+1)*100:.1f}%")

    accuracy = correct / total * 100
    print(f"  [{dataset_name}] Final Accuracy: {accuracy:.2f}% ({correct}/{total})")
    return accuracy, results

In [ ]:
# ============================================================
#  Run Evaluation on All 3 Test Sets
# ============================================================
print(f"Evaluating LoRA-finetuned {ACTIVE_MODEL}...\n")

acc_gsm8k, results_gsm8k = evaluate_model(model, tokenizer, test_gsm8k, "GSM8K")
acc_svamp, results_svamp = evaluate_model(model, tokenizer, test_svamp, "SVAMP")
acc_aqua, results_aqua = evaluate_model(model, tokenizer, test_aqua, "AQuA-RAT")

print(f"\n{'='*50}")
print(f"  PEFT Results for: {ACTIVE_MODEL}")
print(f"  GSM8K:    {acc_gsm8k:.2f}%")
print(f"  SVAMP:    {acc_svamp:.2f}%")
print(f"  AQuA-RAT: {acc_aqua:.2f}%")
print(f"{'='*50}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Evaluating LoRA-finetuned qwen2.5...



In [ ]:
# Save evaluation results
eval_results = {
    "model": cfg["model_name"],
    "method": "LoRA",
    "accuracy": {
        "gsm8k": round(acc_gsm8k, 2),
        "svamp": round(acc_svamp, 2),
        "aqua_rat": round(acc_aqua, 2),
    },
    "lora_config": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "target_modules": cfg["target_modules"],
    },
}

with open(f"/content/eval_results_{ACTIVE_MODEL}.json", "w") as f:
    json.dump(eval_results, f, indent=2)

# Save detailed predictions for error analysis (for teammate LIU YANG)
with open(f"/content/predictions_{ACTIVE_MODEL}_gsm8k.json", "w") as f:
    json.dump(results_gsm8k, f, indent=2)
with open(f"/content/predictions_{ACTIVE_MODEL}_svamp.json", "w") as f:
    json.dump(results_svamp, f, indent=2)
with open(f"/content/predictions_{ACTIVE_MODEL}_aqua.json", "w") as f:
    json.dump(results_aqua, f, indent=2)

print("All results saved!")
print(json.dumps(eval_results, indent=2))